<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

Install the libraries used in this notebook. OpenBB pulls the price data and runs the screener, and the rest handle the table work, statistics, and charts.

In [ ]:
!pip install pandas openbb-terminal scipy matplotlib seaborn

TA-Lib is left out of that command on purpose. It wraps a C library that has to be compiled and installed on your machine first, so pip alone will fail. On most systems you install the C library through conda or your package manager, then run pip install TA-Lib.

## Imports and setup

We use pandas for the price table, openbb for the screener and daily bars, TA-Lib's ATR for the volatility measure, SciPy's spearmanr for the rank correlation, and matplotlib with seaborn for the chart.

In [ ]:
import pandas as pd

In [ ]:
from openbb import obb
from talib import ATR

In [ ]:
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns

## Build a universe of volatile stocks

Pull OpenBB's "most_volatile" screener, then keep only US-listed names priced above $5.

In [ ]:
data = obb.equity.screener(
    provider="yfinance",
    signal="most_volatile",
    metric="technical",
    limit=50,
).to_df()

In [ ]:
universe = data[(data.currency == "USD") & (data.price > 5) & (data.exchange != "PNK")]

The $5 filter matters more than it looks. Sub-$5 stocks have wide gaps between the buy and sell price, so their daily ranges are mostly quoting noise and any volatility measure we build on them describes the spread rather than the stock. We want a universe where the number we calculate reflects actual price movement.

Download daily bars back to 2010 for every ticker in the universe and stack them into one long table.

In [ ]:
stocks = []
for symbol in universe.symbol.tolist():
    df = (
        obb.equity.price.historical(
            symbol=symbol,
            start_date="2020-01-01",
            end_date="2026-08-24",
            interval="1d",
            provider="yfinance",
        ).to_df()
    )
    df["symbol"] = symbol
    stocks.append(df)
prices = pd.concat(stocks)

Dropping "Close" keeps the adjusted close, which already accounts for splits and dividends. If we kept the raw close, a 3-for-1 split would show up as a 67% one-day loss and would distort every return we calculate later. One long table with a ticker column is also the shape pandas wants for per-stock calculations.

Drop any ticker with less than roughly two years of daily bars, then index the table by ticker and date.

In [ ]:
nobs = prices.groupby("symbol").size()
mask = nobs[nobs > 2 * 12 * 21].index
prices = prices[prices.symbol.isin(mask)]

In [ ]:
prices = (
    prices
    .set_index("symbol", append=True)
    .reorder_levels(["symbol", "date"])
).drop_duplicates()

The 2 * 12 * 21 works out to about 504 trading days, since a month has roughly 21 of them. A stock with 60 bars can produce a correlation that looks strong and means nothing, so we cut those before they reach the test. The two-level index of ticker and date is what keeps later groupby calls from running a calculation across the boundary between two different stocks.

## Measure volatility with the ATR indicator

Define a function that calculates the 14-day Average True Range for one stock and rescales it.

In [ ]:
def atr(data):
    df = ATR(data.high, data.low, data.close, timeperiod=14)
    return df.sub(df.mean()).div(df.std())

Average True Range is the average distance between a day's high and low over the last 14 days, so it measures how much a stock swings. Raw ATR is in dollars, which means a $400 stock always looks more volatile than a $8 stock. Subtracting the mean and dividing by the standard deviation puts every stock on the same scale, so a value of 2 means the same thing everywhere.

Run the function separately for each ticker and store the result as a new column.

In [ ]:
prices["atr"] = (
    prices
    .groupby("symbol", group_keys=False)
    .apply(atr)
)

Setting group_keys=False stops pandas from adding an extra index level, so the output lines up with the original rows and drops straight into a column. This is the moment the idea stops being an idea. It's now one number per stock per day, sitting next to the prices, ready to be scored.

## Score the factor against future returns

Calculate trailing returns over six different holding periods for every stock.

In [ ]:
lags = [1, 5, 10, 21, 42, 63]
for lag in lags:
    prices[f"return_{lag}d"] = (
        prices.groupby(level="symbol")
        .close.pct_change(lag)
    )

These are backward-looking returns, measured from some number of days ago up to today. We calculate several lengths because the same number can be built into a one-day test or a three-month test, and we'll reuse these columns in the next step to build the thing we're trying to predict.

Shift each return backward in time so today's row carries the return that happened after today.

In [ ]:
for t in [1, 5, 10, 21]:
    prices[f"target_{t}d"] = (
        prices
        .groupby(level="symbol")[f"return_{t}d"]
        .shift(-t)
    )

The negative shift is the whole test. It puts tomorrow's return on the same row as today's ATR value, so we can ask whether one lines up with the other. Pulling future data backward like this is fine here because we're measuring it, not feeding it to a model. The mistake to avoid is letting a column like target_1d end up as an input, which would make any strategy look brilliant on historical data and fail immediately in live trading.

Plot the standardized ATR against the next day's return.

In [ ]:
target = "target_1d"
metric = "atr"

In [ ]:
j = sns.jointplot(x=metric, y=target, data=prices)
plt.tight_layout()

Look at the shape before you look at the number. A scatter plot tells you whether any relationship runs across the full range of values or comes from a handful of extreme points, which a single correlation figure hides completely.

Measure the rank correlation between the factor and the next day's return, with its p-value.

In [ ]:
df = prices[[metric, target]].dropna()
r, p = spearmanr(df[metric], df[target])
print(f"{r:,.2%} ({p:.2%})")

Spearman compares the order of the values rather than the values themselves, so one stock with a 40% move on an earnings day can't drag the whole result around. Quants call this correlation the information coefficient, and the honest range is small. Anything between 1% and 5% that holds up across different time periods is worth building on, and the p-value tells you how likely you'd see that result from random data alone.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.